#INIT

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType

In [0]:
df = spark.table("data_lakehouse_project.bronze.bronze_prd_info")

In [0]:
RENAME_MAP = {
 "prd_id" : "product_id",
 "prd_key" : "product_code_key",
 "prd_nm" : "product_name",
 "prd_cost" : "product_cost",
 "prd_line" : "product_line",
 "prd_start_dt" : "product_start_date",
 "prd_end_dt" : "product_end_date"
}

#Data transform


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
# Substring prd_key to product category
df = df.withColumn(
    "product_category",
    F.col("prd_key").substr(1, 5)
)

In [0]:
# Substring prd_key to product key
df = df.withColumn(
    "prd_key",
    F.expr("substring(prd_key, 7, len(prd_key))")
)

Date validate

In [0]:
df = df.withColumn(
    "temp_start",F.col("prd_start_dt")
).withColumn(
    "prd_start_dt",
    F.when(F.col("temp_start") > F.col("prd_end_dt"),F.col("prd_end_dt"))
    .otherwise(F.col("temp_start"))
    ).withColumn(
        "prd_end_dt",
        F.when(F.col("temp_start") > F.col("prd_end_dt"),F.col("temp_start"))
        .otherwise(F.col("prd_end_dt"))
    ).drop("temp_start")


Cost handle < 0 value and Null value

In [0]:
df = df.withColumn(
    "prd_cost", 
    F.when(F.col("prd_cost") < 0 , 0)
    .when(F.col("prd_cost").isNull(),0)
    .otherwise(F.col("prd_cost"))
)

Handling null value and nomarlize product_line


In [0]:
df = df.withColumn(
    "prd_line",
    F.when(F.upper(col("prd_line")) == "R", "Road")
    .when(F.upper(col("prd_line")) == "S", "Street")
    .when(F.upper(col("prd_line")) == "M", "Mountain")
    .when(F.upper(col("prd_line")) == "T", "Touring")
    .otherwise("N/a")
)

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("data_lakehouse_project.silver.silver_prd_info")
)

In [0]:
%sql
select 
*
from data_lakehouse_project.silver.silver_prd_info